In [1]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv("data/strong_workouts.csv")


,Date,Workout Name,Duration,Exercise Name,Set Order,Weight,Reps,Distance,Seconds,Notes,Workout Notes,RPE
0,2025-08-26 10:44:22,Morning Workout,28m,Seated Overhead Press (Dumbbell),1,17.5,10.0,0.0,0.0,NaN,NaN,NaN
1,2025-08-26 10:44:22,Morning Workout,28m,Seated Overhead Press (Dumbbell),Rest Timer,0.0,0.0,0.0,120.0,NaN,NaN,NaN
2,2025-08-26 10:44:22,Morning Workout,28m,Seated Overhead Press (Dumbbell),2,17.5,10.0,0.0,0.0,NaN,NaN,NaN
3,2025-08-26 10:44:22,Morning Workout,28m,Seated Overhead Press (Dumbbell),Rest Timer,0.0,0.0,0.0,120.0,NaN,NaN,NaN
4,2025-08-26 10:44:22,Morning Workout,28m,Seated Overhead Press (Dumbbell),3,17.5,10.0,0.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
5016,2026-05-09 15:19:09,Back and Biceps (Pull),30m,Hammer Curl (Dumbbell),Rest Timer,0.0,0.0,0.0,120.0,NaN,NaN,NaN
5017,2026-05-09 15:19:09,Back and Biceps (Pull),30m,Hammer Curl (Dumbbell),3,20.0,12.0,0.0,0.0,NaN,NaN,NaN
5018,2026-05-09 15:19:09,Back and Biceps (Pull),30m,Hammer Curl (Dumbbell),Rest Timer,0.0,0.0,0.0,120.0,NaN,NaN,NaN
5019,2026-05-09 15:19:09,Back and Biceps (Pull),30m,Hammer Curl (Dumbbell),4,20.0,12.0,0.0,0.0,NaN,NaN,NaN


In [24]:
clean_data = data.copy()
clean_data = clean_data.dropna(axis=1)

clean_data = clean_data[clean_data["Set Order"].str.fullmatch(r"\d+")].copy()

clean_data["Date"] = pd.to_datetime(clean_data["Date"])
clean_data["Weight"] = pd.to_numeric(clean_data["Weight"], errors="coerce")

clean_data["Reps"] = pd.to_numeric(clean_data["Reps"], errors="coerce")
clean_data["Volume"] = clean_data["Reps"] * clean_data["Weight"]
clean_data["Estimate_1RM"] = clean_data["Weight"] * (1 + clean_data["Reps"]/30)

clean_data["Week_Start"] = clean_data["Date"].dt.to_period("W").apply(lambda x: x.start_time)
clean_data

,Date,Workout Name,Duration,Exercise Name,Set Order,Weight,Reps,Distance,Seconds,Volume,Estimate_1RM,Week_Start
0,2025-08-26 10:44:22,Morning Workout,28m,Seated Overhead Press (Dumbbell),1,17.5,10.0,0.0,0.0,175.0,23.333333,2025-08-25
2,2025-08-26 10:44:22,Morning Workout,28m,Seated Overhead Press (Dumbbell),2,17.5,10.0,0.0,0.0,175.0,23.333333,2025-08-25
4,2025-08-26 10:44:22,Morning Workout,28m,Seated Overhead Press (Dumbbell),3,17.5,10.0,0.0,0.0,175.0,23.333333,2025-08-25
6,2025-08-26 10:44:22,Morning Workout,28m,Bench Press (Dumbbell),1,17.5,10.0,0.0,0.0,175.0,23.333333,2025-08-25
8,2025-08-26 10:44:22,Morning Workout,28m,Bench Press (Dumbbell),2,17.5,10.0,0.0,0.0,175.0,23.333333,2025-08-25
...,...,...,...,...,...,...,...,...,...,...,...,...
5011,2026-05-09 15:19:09,Back and Biceps (Pull),30m,Bicep Curl (Dumbbell),3,35.0,8.0,0.0,0.0,280.0,44.333333,2026-05-04
5013,2026-05-09 15:19:09,Back and Biceps (Pull),30m,Hammer Curl (Dumbbell),1,20.0,12.0,0.0,0.0,240.0,28.000000,2026-05-04
5015,2026-05-09 15:19:09,Back and Biceps (Pull),30m,Hammer Curl (Dumbbell),2,20.0,12.0,0.0,0.0,240.0,28.000000,2026-05-04
5017,2026-05-09 15:19:09,Back and Biceps (Pull),30m,Hammer Curl (Dumbbell),3,20.0,12.0,0.0,0.0,240.0,28.000000,2026-05-04


In [27]:
workouts_per_week = clean_data.groupby("Week_Start")["Date"].nunique().reset_index(name="Workouts")
workouts_per_week


,Week_Start,Workouts
0,2025-08-25,6
1,2025-09-01,4
2,2025-09-08,5
3,2025-09-15,7
4,2025-09-22,7
5,2025-09-29,6
6,2025-10-06,6
7,2025-10-13,6
8,2025-10-20,7
9,2025-10-27,6


In [ ]:
daily_strength = clean_data.groupby(["Date", "Exercise Name"])["Estimate_1RM"].max().reset_index()

daily_strength


,Date,Exercise Name,Estimate_1RM
0,2025-08-26 10:44:22,Bench Press (Dumbbell),23.333333
1,2025-08-26 10:44:22,Bicep Curl (Dumbbell),24.500000
2,2025-08-26 10:44:22,Lateral Raise (Dumbbell),9.500000
3,2025-08-26 10:44:22,Plank,0.000000
4,2025-08-26 10:44:22,Seated Overhead Press (Dumbbell),23.333333
...,...,...,...
660,2026-05-08 15:33:04,Triceps Extension (Dumbbell),26.666667
661,2026-05-08 15:33:04,Triceps Pushdown (Cable - Straight Bar),132.000000
662,2026-05-09 15:19:09,Bicep Curl (Dumbbell),44.333333
663,2026-05-09 15:19:09,Hammer Curl (Dumbbell),28.000000


In [31]:
#TEST CELL FOR DAILY STRENGTH
exercise = "Bicep Curl (Dumbbell)" 

bench_trend = daily_strength[daily_strength["Exercise Name"] == exercise]

bench_trend

,Date,Exercise Name,Estimate_1RM
1,2025-08-26 10:44:22,Bicep Curl (Dumbbell),24.500000
7,2025-08-27 18:09:55,Bicep Curl (Dumbbell),26.666667
15,2025-08-30 11:45:20,Bicep Curl (Dumbbell),27.333333
26,2025-09-03 09:12:19,Bicep Curl (Dumbbell),20.000000
35,2025-09-09 14:07:00,Bicep Curl (Dumbbell),21.500000
...,...,...,...
634,2026-04-18 21:19:52,Bicep Curl (Dumbbell),42.000000
640,2026-04-22 16:28:58,Bicep Curl (Dumbbell),44.333333
646,2026-04-25 23:12:57,Bicep Curl (Dumbbell),42.000000
649,2026-04-29 20:49:19,Bicep Curl (Dumbbell),44.333333
